<a href="https://colab.research.google.com/github/SANGHATI23/genomic-evidence-reliability/blob/main/08_GES_Aware_Genomic_RAG_Cell_7C1_Quality_Reranking_and_Top5_Execution_Authorization_V2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

ROOT = Path('/content/drive/MyDrive/GES_RAG_Temporal_Study')
if not ROOT.exists():
    raise FileNotFoundError(f'Project root does not exist: {ROOT}')

print(f'Project root: {ROOT}')


Mounted at /content/drive
Project root: /content/drive/MyDrive/GES_RAG_Temporal_Study


## 1. Imports, frozen identities, paths, and overwrite protection

In [2]:
from __future__ import annotations

from collections import OrderedDict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any
import csv
import hashlib
import json
import re

import pyarrow.parquet as pq

NOTEBOOK_NAME = '08_GES_Aware_Genomic_RAG_Cell_7C1_Quality_Reranking_and_Top5_Execution_Authorization_V2.ipynb'
CELL_ID = '7C1'
STAGE = '7C'
PACKAGE_VERSION = 'v1'
CREATED_UTC = datetime.now(timezone.utc).isoformat()

EXPECTED_CELL_7C0_DECISION = (
    'PASS_STAGE7C0_EXACT_PUBMEDBERT_REVISION_CORPUS_AND_QUESTION_EMBEDDINGS_FLOAT32_L2_768_'
    'EXACT_FAISS_INDEXFLATIP_COMMON_SEMANTIC_TOP20_80_QUESTIONS_1600_CANDIDATES_MATERIALIZED_'
    'CHECKSUM_PROTECTED_NO_CELL7A3_SCORES_QUALITY_RERANKING_RRF_TOP5_PROMPTS_LLM_ANSWER_KEY_'
    'OUTCOME_INSPECTION_ADJUDICATION_OR_RAG_METRICS_NEXT_EXECUTION_NOT_AUTHORIZED'
)

EXPECTED_CELL_7A3_DECISION = (
    'PASS_STAGE7A3_FROZEN_T1_FULL_GES_NO_STAR_GES_AND_COMBINED_METADATA_SCORES_MATERIALIZED_'
    'CHECKSUM_PROTECTED_T0_PREDICT_PROBA_REPRODUCED_NO_FITTING_NO_THRESHOLDING_NO_RANKING_'
    'NO_RAG_CORPUS_EMBEDDINGS_OR_LLM_NEXT_STAGE_REQUIRES_SEPARATE_PROTOCOL_FREEZE'
)

# Exact successful Cell 7C0 package reported by the terminal PASS.
CELL_7C0_DIR = (
    ROOT / 'outputs' / 'rag_execution' / 'stage7_rag'
    / 'cell_7c0_embedding_and_semantic_retrieval_v1'
)
CELL_7C0_QC_DIR = (
    ROOT / 'outputs' / 'quality_checks' / 'stage7_rag'
    / 'cell_7c0_embedding_and_semantic_retrieval_v1'
)
CELL_7C0_CONFIG_DIR = (
    ROOT / 'configs' / 'stage7_rag'
    / 'cell_7c0_embedding_and_semantic_retrieval_v1'
)

EXPECTED_CELL_7C0 = OrderedDict([
    ('model_artifact_inventory', {
        'path': CELL_7C0_DIR / 'cell_7c0_embedding_model_artifact_inventory_v1.json',
        'sha256': '1bdca6adcfbb31eb81d928bf103c5821d42e93695653debcec6041c442064889',
    }),
    ('corpus_embeddings', {
        'path': CELL_7C0_DIR / 'cell_7c0_semantic_corpus_embeddings_float32_l2_v1.npy',
        'sha256': '3f9360f39c8730131e0784d82cb6a72a4dc14b37e4e177cc0368cfbffed1df9a',
    }),
    ('corpus_embedding_identity', {
        'path': CELL_7C0_DIR / 'cell_7c0_semantic_corpus_embedding_identity_v1.parquet',
        'sha256': '8c853afc5f6db4d54dc9d73d1b24dbece5e13bbfa891db978f6f5ec42782153e',
    }),
    ('question_embeddings', {
        'path': CELL_7C0_DIR / 'cell_7c0_primary_question_embeddings_float32_l2_v1.npy',
        'sha256': '163394b376eda504d516ad6d74c01d79f48a247785aa909a6e3b1f8d7be36ff2',
    }),
    ('question_embedding_identity', {
        'path': CELL_7C0_DIR / 'cell_7c0_primary_question_embedding_identity_v1.csv',
        'sha256': '8dff5943f4e109c6d500763c803eaaeea1e8e4f67bdd3b270687acd2ec655873',
    }),
    ('faiss_index', {
        'path': CELL_7C0_DIR / 'cell_7c0_semantic_corpus_indexflatip_v1.faiss',
        'sha256': 'f676bca91e1c3dcdccb69eb76d626bf161cf81ae0eba05824e21141ddaf3718a',
    }),
    ('semantic_top20_pool', {
        'path': CELL_7C0_DIR / 'cell_7c0_common_semantic_top20_candidate_pool_v1.parquet',
        'sha256': 'be7f7cfc369952fc905cde5fb843e126b8e36d86f9bc625be853e9489d5327cb',
    }),
    ('execution_report', {
        'path': CELL_7C0_DIR / 'cell_7c0_embedding_and_semantic_retrieval_report_v1.json',
        'sha256': 'cf2402354699f0e69aa547efb67655145673d33de5fcb7c86281afc32c7282e2',
    }),
    ('qc', {
        'path': CELL_7C0_QC_DIR / 'cell_7c0_embedding_and_semantic_retrieval_qc_v1.json',
        'sha256': '732c1155fb49e95da8597b9d94d5c3f0d3da56b09dd9ec765758c5d9983e529a',
    }),
    ('manifest', {
        'path': CELL_7C0_CONFIG_DIR / 'cell_7c0_embedding_and_semantic_retrieval_manifest_v1.json',
        'sha256': '3b645237d8d5dfa04345f649b3dd2446e94abf10fb9e855e2c98d642cdd837e8',
    }),
])

# Frozen Cell 7A3 score package. This authorization verifies bytes/metadata only.
CELL_7A3_SCORE_TABLE = (
    ROOT / 'data_processed' / 'stage7_rag'
    / 'cell_7a3_t1_frozen_ges_and_metadata_scores_v1.parquet'
)
CELL_7A3_MANIFEST = (
    ROOT / 'configs' / 'stage7_rag'
    / 'cell_7a3_t1_score_materialization_manifest_v1.json'
)
EXPECTED_CELL_7A3_SCORE_SHA256 = 'e9b162c9add5aed34a4d68d8bf625251293a18c549945468e627a498843eb802'
EXPECTED_CELL_7A3_MANIFEST_SHA256 = '99bcff934f5e0c15d8357450fb3992eccfeb972ab61ddba8431a843c13b532dd'

# Frozen Cell 7B4 downstream execution configuration.
CELL_7B4_DIR = ROOT / 'configs' / 'stage7_rag' / 'cell_7b4_configuration_freeze_v1'
CELL_7B4_QC_DIR = ROOT / 'outputs' / 'quality_checks' / 'stage7_rag' / 'cell_7b4_configuration_freeze_v1'

EMBEDDING_RETRIEVAL_CONFIG = CELL_7B4_DIR / 'cell_7b4_embedding_retrieval_configuration_v1.json'
QUALITY_CONFIG = CELL_7B4_DIR / 'cell_7b4_quality_reranking_configuration_v1.json'
CONDITION_ALIASES = CELL_7B4_DIR / 'cell_7b4_condition_alias_inventory_v1.csv'
CELL_7B4_MANIFEST = CELL_7B4_DIR / 'cell_7b4_configuration_freeze_manifest_v1.json'

EXPECTED_EMBEDDING_RETRIEVAL_CONFIG_SHA256 = 'ab209e48b025652e5ddbb79891225334ffc51de62f157b430c21aef30865b807'
EXPECTED_QUALITY_CONFIG_SHA256 = '35c3871db6dd6bad9436e7202a67b064cc98beee06d19894e409d42f7ca006fb'
EXPECTED_CONDITION_ALIASES_SHA256 = '6eb45683b42a456d2b6788a5fcf6b9cd95fc606afe9627610ebbc11914312cb9'
EXPECTED_CELL_7B4_MANIFEST_SHA256 = '18a5d6cb3cadca0eab950839a19022686fc6bad2c398ed87f2a48c66eff462fe'

EXPECTED_CELL_7B4_DECISION = (
    'PASS_STAGE7B4_EXACT_EMBEDDING_MODEL_REVISION_TEXT_NORMALIZATION_SIMILARITY_TOP20_'
    'CANDIDATE_POOL_TOP5_CONTEXT_QUALITY_RERANKING_BLINDED_ALIASES_PROMPTS_STRICT_'
    'RESPONSE_SCHEMA_FIXED_LLM_SNAPSHOT_GENERATION_RUNTIME_AND_DETERMINISTIC_CONTROLS_'
    'FROZEN_CHECKSUM_PROTECTED_NO_EMBEDDINGS_RETRIEVAL_RERANKING_PROMPT_MATERIALIZATION_'
    'LLM_ANSWER_KEY_OUTCOME_INSPECTION_OR_RAG_EVALUATION_EXECUTION_NOT_AUTHORIZED'
)

AUTH_DIR = ROOT / 'configs' / 'stage7_rag' / 'cell_7c1_quality_reranking_authorization_v1'
QC_DIR = ROOT / 'outputs' / 'quality_checks' / 'stage7_rag' / 'cell_7c1_quality_reranking_authorization_v1'

OUTPUTS = OrderedDict([
    ('authorization', AUTH_DIR / 'cell_7c1_stage7c_cell7c2_quality_reranking_top5_execution_authorization_v1.json'),
    ('input_inventory', AUTH_DIR / 'cell_7c1_authorized_quality_reranking_input_inventory_v1.csv'),
    ('qc', QC_DIR / 'cell_7c1_quality_reranking_authorization_qc_v1.json'),
    ('manifest', AUTH_DIR / 'cell_7c1_quality_reranking_authorization_manifest_v1.json'),
])

for directory in (AUTH_DIR, QC_DIR):
    directory.mkdir(parents=True, exist_ok=True)

if OUTPUTS['manifest'].exists():
    raise FileExistsError(
        f"Cell 7C1 manifest already exists: {OUTPUTS['manifest']}\n"
        'Fail-closed overwrite protection is active.'
    )

print(f'Authorization directory: {AUTH_DIR}')
print(f'QC directory           : {QC_DIR}')


Authorization directory: /content/drive/MyDrive/GES_RAG_Temporal_Study/configs/stage7_rag/cell_7c1_quality_reranking_authorization_v1
QC directory           : /content/drive/MyDrive/GES_RAG_Temporal_Study/outputs/quality_checks/stage7_rag/cell_7c1_quality_reranking_authorization_v1


## 2. Strict checksum, sidecar, JSON, and metadata helpers

In [3]:
def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    path = Path(path)
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for block in iter(lambda: handle.read(chunk_size), b''):
            digest.update(block)
    return digest.hexdigest()


def sidecar_path(path: Path) -> Path:
    return Path(str(path) + '.sha256')


def read_sidecar_hash(path: Path) -> str:
    text = Path(path).read_text(encoding='utf-8').strip()
    matches = re.findall(r'\b[a-fA-F0-9]{64}\b', text)
    if not matches:
        raise ValueError(f'No SHA-256 found in sidecar: {path}')
    return matches[0].lower()


def sidecar_is_valid(path: Path) -> bool:
    path = Path(path)
    sc = sidecar_path(path)
    return (
        path.exists()
        and sc.exists()
        and read_sidecar_hash(sc) == sha256_file(path)
    )


def verify_exact_artifact(label: str, path: Path, expected_hash: str) -> dict[str, Any]:
    if not path.exists():
        raise FileNotFoundError(f'Missing {label}: {path}')
    observed = sha256_file(path)
    if observed != expected_hash:
        raise AssertionError(
            f'SHA-256 mismatch for {label}\n'
            f'Expected: {expected_hash}\nObserved: {observed}\nPath: {path}'
        )
    if not sidecar_is_valid(path):
        raise AssertionError(f'Invalid or missing SHA-256 sidecar for {label}: {path}')
    return {
        'label': label,
        'path': str(path),
        'sha256': observed,
        'sidecar_path': str(sidecar_path(path)),
        'sidecar_valid': True,
    }


def to_json_native(value: Any) -> Any:
    if value is None or isinstance(value, (str, int, float, bool)):
        return value
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, dict):
        return {str(k): to_json_native(v) for k, v in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [to_json_native(v) for v in value]
    if hasattr(value, 'item'):
        return to_json_native(value.item())
    raise TypeError(f'Unsupported JSON value: {value.__class__.__name__}')


def stable_write_json(path: Path, payload: Any) -> str:
    path.parent.mkdir(parents=True, exist_ok=True)
    native = to_json_native(payload)
    data = (
        json.dumps(native, indent=2, sort_keys=True, ensure_ascii=False, allow_nan=False)
        + '\n'
    ).encode('utf-8')
    path.write_bytes(data)
    return sha256_file(path)


def stable_write_csv(path: Path, rows: list[dict[str, Any]], fieldnames: list[str]) -> str:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', encoding='utf-8', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames, lineterminator='\n')
        writer.writeheader()
        for row in rows:
            writer.writerow({key: to_json_native(row.get(key, '')) for key in fieldnames})
    return sha256_file(path)


def write_sidecar(path: Path) -> None:
    digest = sha256_file(path)
    sidecar_path(path).write_text(f'{digest}  {path.name}\n', encoding='utf-8')


def parquet_metadata(path: Path) -> dict[str, Any]:
    pf = pq.ParquetFile(path)
    return {
        'rows': int(pf.metadata.num_rows),
        'columns': int(pf.metadata.num_columns),
        'schema_names': list(pf.schema_arrow.names),
    }


def load_json(path: Path) -> dict[str, Any]:
    return json.loads(path.read_text(encoding='utf-8'))


print('Strict helpers loaded.')


Strict helpers loaded.


## 3. Reverify the complete frozen Cell 7C0 package

In [4]:
verified_7c0 = OrderedDict()

for artifact_id, spec in EXPECTED_CELL_7C0.items():
    verified_7c0[artifact_id] = verify_exact_artifact(
        f'Cell 7C0 {artifact_id}',
        spec['path'],
        spec['sha256'],
    )

cell_7c0_manifest = load_json(EXPECTED_CELL_7C0['manifest']['path'])
cell_7c0_qc = load_json(EXPECTED_CELL_7C0['qc']['path'])
cell_7c0_report = load_json(EXPECTED_CELL_7C0['execution_report']['path'])

observed_7c0_decision = (
    cell_7c0_manifest.get('terminal_decision')
    or cell_7c0_manifest.get('decision')
    or cell_7c0_report.get('terminal_decision')
    or cell_7c0_report.get('decision')
)

if observed_7c0_decision != EXPECTED_CELL_7C0_DECISION:
    raise AssertionError(
        'Cell 7C0 terminal decision mismatch.\n'
        f'Observed: {observed_7c0_decision}'
    )

top20_meta = parquet_metadata(EXPECTED_CELL_7C0['semantic_top20_pool']['path'])
if top20_meta['rows'] != 1600:
    raise AssertionError(f"Cell 7C0 top-20 row count changed: {top20_meta['rows']}")

qc_failed = cell_7c0_qc.get('failed_checks', 0)
if isinstance(qc_failed, list):
    qc_failed_count = len(qc_failed)
else:
    qc_failed_count = int(qc_failed)
if qc_failed_count != 0:
    raise AssertionError(f'Cell 7C0 QC contains failures: {qc_failed}')

print(f'Cell 7C0 artifacts                    : {len(verified_7c0)}/10 exact hashes + sidecars')
print('Cell 7C0 terminal PASS               : VERIFIED')
print(f'Common semantic top-20 rows          : {top20_meta["rows"]:,}')
print('Cell 7A3 score rows opened           : NO')


Cell 7C0 artifacts                    : 10/10 exact hashes + sidecars
Cell 7C0 terminal PASS               : VERIFIED
Common semantic top-20 rows          : 1,600
Cell 7A3 score rows opened           : NO


## 4. Reverify score/configuration inputs without loading row-level Cell 7A3 scores

In [5]:
score_table_record = verify_exact_artifact(
    'Cell 7A3 frozen T1 score table',
    CELL_7A3_SCORE_TABLE,
    EXPECTED_CELL_7A3_SCORE_SHA256,
)
score_manifest_record = verify_exact_artifact(
    'Cell 7A3 manifest',
    CELL_7A3_MANIFEST,
    EXPECTED_CELL_7A3_MANIFEST_SHA256,
)
embedding_retrieval_record = verify_exact_artifact(
    'Cell 7B4 embedding/retrieval configuration',
    EMBEDDING_RETRIEVAL_CONFIG,
    EXPECTED_EMBEDDING_RETRIEVAL_CONFIG_SHA256,
)
quality_config_record = verify_exact_artifact(
    'Cell 7B4 quality-reranking configuration',
    QUALITY_CONFIG,
    EXPECTED_QUALITY_CONFIG_SHA256,
)
condition_alias_record = verify_exact_artifact(
    'Cell 7B4 condition aliases',
    CONDITION_ALIASES,
    EXPECTED_CONDITION_ALIASES_SHA256,
)
cell_7b4_manifest_record = verify_exact_artifact(
    'Cell 7B4 manifest',
    CELL_7B4_MANIFEST,
    EXPECTED_CELL_7B4_MANIFEST_SHA256,
)

cell_7a3_manifest = load_json(CELL_7A3_MANIFEST)
observed_7a3_decision = (
    cell_7a3_manifest.get('terminal_decision')
    or cell_7a3_manifest.get('decision')
)
if observed_7a3_decision != EXPECTED_CELL_7A3_DECISION:
    raise AssertionError(f'Cell 7A3 terminal decision mismatch: {observed_7a3_decision}')

cell_7b4_manifest = load_json(CELL_7B4_MANIFEST)
observed_7b4_decision = (
    cell_7b4_manifest.get('terminal_decision')
    or cell_7b4_manifest.get('decision')
)
if observed_7b4_decision != EXPECTED_CELL_7B4_DECISION:
    raise AssertionError(
        'Cell 7B4 terminal decision mismatch.\n'
        f'Observed: {observed_7b4_decision}'
    )

# Metadata only: no row-group or row-level score content is read.
score_meta = parquet_metadata(CELL_7A3_SCORE_TABLE)
if score_meta['rows'] != 100_920:
    raise AssertionError(f'Cell 7A3 score-table row count changed: {score_meta["rows"]}')
if score_meta['columns'] != 28:
    raise AssertionError(f'Cell 7A3 score-table column count changed: {score_meta["columns"]}')

embedding_retrieval_config = load_json(EMBEDDING_RETRIEVAL_CONFIG)
quality_config = load_json(QUALITY_CONFIG)

# Read aliases because aliases are configuration, not score/outcome content.
with CONDITION_ALIASES.open('r', encoding='utf-8', newline='') as handle:
    alias_rows = list(csv.DictReader(handle))
if len(alias_rows) != 6:
    raise AssertionError(f'Expected six frozen blinded aliases; observed {len(alias_rows)}.')

# ------------------------------------------------------------------
# Exact structural validation of the Cell 7B4 frozen configuration.
# Do NOT search for display strings such as "top-5". The frozen JSON
# represents final top-5 numerically as final_context_k = 5.
# ------------------------------------------------------------------
candidate_pool = embedding_retrieval_config.get('candidate_pool', {})
rerank_pool = quality_config.get('candidate_pool_invariance', {})
rrf = quality_config.get('weighted_reciprocal_rank_fusion', {})
random_control = quality_config.get('random_quality_control', {})
condition_a = quality_config.get('condition_a_semantic_only', {})
conditions_b_to_f = quality_config.get('conditions_b_to_f', {})
condition_inventory = quality_config.get('condition_inventory', [])

config_checks = OrderedDict([
    ('cell_7b4_terminal_pass_exact', observed_7b4_decision == EXPECTED_CELL_7B4_DECISION),

    ('embedding_semantic_candidate_pool_k_20',
     candidate_pool.get('semantic_candidate_pool_k') == 20),
    ('embedding_final_context_k_5',
     candidate_pool.get('final_context_k') == 5),
    ('embedding_same_top20_pool_for_all_conditions',
     candidate_pool.get('same_top20_pool_for_all_conditions') is True),
    ('embedding_candidate_pool_reused_without_mutation',
     candidate_pool.get('candidate_pool_reused_without_mutation') is True),
    ('embedding_hard_exclusion_false',
     candidate_pool.get('hard_evidence_exclusion') is False),

    ('rerank_input_pool_exact',
     rerank_pool.get('input_pool') == 'same frozen semantic top-20 for all six conditions'),
    ('rerank_input_order_semantic_ascending',
     rerank_pool.get('input_order') == 'semantic rank ascending'),
    ('rerank_final_context_k_5',
     rerank_pool.get('final_context_k') == 5),
    ('rerank_hard_exclusion_false',
     rerank_pool.get('hard_exclusion') is False),
    ('rerank_quality_score_not_exposed_to_llm',
     rerank_pool.get('quality_score_exposed_to_llm') is False),

    ('rrf_formula_exact',
     rrf.get('formula') == '0.75/(60 + semantic_rank) + 0.25/(60 + quality_rank)'),
    ('rrf_semantic_weight_075',
     rrf.get('semantic_weight') == 0.75),
    ('rrf_quality_weight_025',
     rrf.get('quality_weight') == 0.25),
    ('rrf_constant_60',
     rrf.get('rrf_constant') == 60),
    ('rrf_descending_sort',
     rrf.get('descending_sort') is True),

    ('condition_a_no_quality_rank',
     condition_a.get('quality_rank_constructed') is False),
    ('condition_a_no_rrf',
     condition_a.get('rrf_applied') is False),
    ('condition_a_final_context_semantic_1_to_5',
     condition_a.get('final_context') == 'semantic ranks 1 through 5'),

    ('conditions_b_to_f_rrf_applied',
     conditions_b_to_f.get('rrf_applied') is True),
    ('conditions_b_to_f_final_context_top_five',
     conditions_b_to_f.get('final_context') == 'top five by frozen weighted RRF'),

    ('random_quality_seed_20260722',
     random_control.get('seed') == 20260722),
    ('random_quality_hash_sha256',
     random_control.get('hash_algorithm') == 'SHA-256'),
    ('random_quality_payload_exact',
     random_control.get('payload') == '20260722|{question_id}|{rcv_accession}'),

    ('six_condition_inventory_rows',
     len(condition_inventory) == 6),
    ('primary_comparison_d_vs_a',
     quality_config.get('primary_comparison') == 'D_vs_A'),
])

failed_config_checks = [
    name for name, passed in config_checks.items()
    if not bool(passed)
]
if failed_config_checks:
    raise RuntimeError(
        'Cell 7B4 structural configuration validation failed:\n- '
        + '\n- '.join(failed_config_checks)
    )

print(f'Cell 7A3 score metadata               : {score_meta["rows"]:,} rows × {score_meta["columns"]} columns')
print('Cell 7A3 terminal PASS               : VERIFIED')
print('Cell 7B4 terminal PASS               : VERIFIED')
print('Cell 7B4 top-20                      : 20 (structurally verified)')
print('Cell 7B4 final context               : 5 (structurally verified)')
print('Cell 7B4 RRF                         : 0.75 semantic + 0.25 quality; constant 60')
print('Cell 7B4 hard exclusion              : FALSE')
print(f'Frozen blinded condition aliases     : {len(alias_rows)}')
print('Cell 7A3 row-level scores loaded     : NO')
print('Answer-key outcomes inspected        : NO')


Cell 7A3 score metadata               : 100,920 rows × 28 columns
Cell 7A3 terminal PASS               : VERIFIED
Cell 7B4 terminal PASS               : VERIFIED
Cell 7B4 top-20                      : 20 (structurally verified)
Cell 7B4 final context               : 5 (structurally verified)
Cell 7B4 RRF                         : 0.75 semantic + 0.25 quality; constant 60
Cell 7B4 hard exclusion              : FALSE
Frozen blinded condition aliases     : 6
Cell 7A3 row-level scores loaded     : NO
Answer-key outcomes inspected        : NO


## 5. Freeze the narrow Cell 7C2 quality-reranking/top-5 execution authorization

In [6]:
AUTHORIZED_CELL = {
    'stage': '7C',
    'cell_id': '7C2',
    'title': 'Frozen six-condition quality ranking, fixed RRF reranking, and final top-5 materialization',
    'authorized_once': True,
    'overwrite_existing_outputs': False,
}

AUTHORIZED_OPERATIONS = [
    'Load the exact checksum-frozen Cell 7C0 common semantic top-20 candidate pool only as the retrieval candidate set.',
    'Load the exact checksum-frozen Cell 7A3 T1 score table after reverifying its hash and sidecar.',
    'Join score/configuration fields by the frozen RCV accession identity without changing the candidate pool.',
    'Construct Condition A semantic-only ordering from the frozen semantic ranking.',
    'Construct Condition B review/conflict-aware quality ranking using the exact Cell 7B4 frozen formula.',
    'Construct Condition C combined-metadata quality ranking using the exact Cell 7B4 frozen formula.',
    'Construct Condition D Full-GES quality ranking using the exact Cell 7B4 frozen formula.',
    'Construct Condition E no-star-GES quality ranking using the exact Cell 7B4 frozen formula.',
    'Construct Condition F deterministic random-quality ranking using frozen seed 20260722 and the Cell 7B4 algorithm.',
    'Apply the exact frozen reciprocal-rank-fusion rule: 0.75 semantic + 0.25 quality, RRF constant 60.',
    'Materialize exactly five final context records per question per condition.',
    'Freeze condition-specific ranking/top-5 artifacts with SHA-256 sidecars, QC, execution report, and manifest.',
]

PROHIBITED_OPERATIONS = [
    'Generate new embeddings or rerun semantic retrieval.',
    'Add, remove, replace, or expand any record in the frozen common semantic top-20 candidate pool.',
    'Change top-20, final top-5, 0.75/0.25 fusion weights, RRF constant 60, condition definitions, aliases, or random seed.',
    'Use any hard evidence exclusion.',
    'Fit, refit, recalibrate, retune, threshold-optimize, or otherwise modify GES or comparator scores.',
    'Expose Full-GES, no-star-GES, metadata-quality, random-quality, semantic-rank, quality-rank, or RRF values to the LLM as answer content.',
    'Materialize prompts.',
    'Call an LLM.',
    'Load or inspect structured answer-key outcomes.',
    'Perform adjudication.',
    'Calculate retrieval, answer, RAG, bootstrap, or statistical performance metrics.',
]

authorization_decision = (
    'AUTHORIZE_STAGE7C_CELL7C2_FROZEN_CELL7A3_SCORE_LOADING_SIX_CONDITION_QUALITY_RANKING_'
    'FIXED_075_SEMANTIC_025_QUALITY_RRF_CONSTANT60_AND_FINAL_TOP5_MATERIALIZATION_ONLY_'
    'NO_NEW_RETRIEVAL_HARD_EXCLUSION_PROMPTS_LLM_ANSWER_KEYS_ADJUDICATION_OR_RAG_METRICS'
)

input_records = []
for artifact_id in ('semantic_top20_pool', 'manifest', 'qc', 'execution_report'):
    record = verified_7c0[artifact_id]
    input_records.append({
        'input_id': f'cell_7c0_{artifact_id}',
        'source_cell': '7C0',
        'path': record['path'],
        'sha256': record['sha256'],
        'sidecar_valid': True,
        'row_level_content_opened_in_7c1': False,
    })

for input_id, source_cell, record in [
    ('cell_7a3_score_table', '7A3', score_table_record),
    ('cell_7a3_manifest', '7A3', score_manifest_record),
    ('cell_7b4_embedding_retrieval_config', '7B4', embedding_retrieval_record),
    ('cell_7b4_quality_config', '7B4', quality_config_record),
    ('cell_7b4_condition_aliases', '7B4', condition_alias_record),
    ('cell_7b4_manifest', '7B4', cell_7b4_manifest_record),
]:
    input_records.append({
        'input_id': input_id,
        'source_cell': source_cell,
        'path': record['path'],
        'sha256': record['sha256'],
        'sidecar_valid': True,
        'row_level_content_opened_in_7c1': False,
    })

authorization_payload = {
    'cell_id': CELL_ID,
    'stage': STAGE,
    'package_version': PACKAGE_VERSION,
    'created_utc': CREATED_UTC,
    'notebook': NOTEBOOK_NAME,
    'project_root': str(ROOT),
    'authorization_type': 'fail_closed_scientific_execution_authorization',
    'authorization_decision': authorization_decision,
    'authorized_cell': AUTHORIZED_CELL,
    'authorized_operations': AUTHORIZED_OPERATIONS,
    'prohibited_operations': PROHIBITED_OPERATIONS,
    'frozen_execution_design': {
        'conditions': 6,
        'common_semantic_candidate_pool': 20,
        'final_context_records_per_question_condition': 5,
        'questions': 80,
        'expected_condition_question_pairs': 480,
        'expected_final_top5_rows': 2400,
        'semantic_weight': 0.75,
        'quality_weight': 0.25,
        'rrf_constant': 60,
        'hard_exclusion': False,
        'random_quality_seed': 20260722,
        'scores_exposed_to_llm': False,
    },
    'verified_inputs': input_records,
    'cell_7a3_score_rows_opened': False,
    'answer_key_outcomes_inspected': False,
    'prompts_materialized': False,
    'llm_called': False,
    'rag_metrics_calculated': False,
}
stable_write_json(OUTPUTS['authorization'], authorization_payload)
write_sidecar(OUTPUTS['authorization'])

stable_write_csv(
    OUTPUTS['input_inventory'],
    input_records,
    [
        'input_id',
        'source_cell',
        'path',
        'sha256',
        'sidecar_valid',
        'row_level_content_opened_in_7c1',
    ],
)
write_sidecar(OUTPUTS['input_inventory'])

print(f'Authorization decision: {authorization_decision}')
print('Cell 7A3 score rows opened: NO')
print('Cell 7C2 scientific execution performed: NO')


Authorization decision: AUTHORIZE_STAGE7C_CELL7C2_FROZEN_CELL7A3_SCORE_LOADING_SIX_CONDITION_QUALITY_RANKING_FIXED_075_SEMANTIC_025_QUALITY_RRF_CONSTANT60_AND_FINAL_TOP5_MATERIALIZATION_ONLY_NO_NEW_RETRIEVAL_HARD_EXCLUSION_PROMPTS_LLM_ANSWER_KEYS_ADJUDICATION_OR_RAG_METRICS
Cell 7A3 score rows opened: NO
Cell 7C2 scientific execution performed: NO


## 6. QC, manifest, immutable readback, and terminal decision

In [7]:
checks = OrderedDict([
    ('cell_7c0_all_10_hashes_exact', len(verified_7c0) == 10),
    ('cell_7c0_all_10_sidecars_valid', all(v['sidecar_valid'] for v in verified_7c0.values())),
    ('cell_7c0_terminal_pass_exact', observed_7c0_decision == EXPECTED_CELL_7C0_DECISION),
    ('cell_7c0_top20_rows_1600', top20_meta['rows'] == 1600),
    ('cell_7a3_score_hash_exact', sha256_file(CELL_7A3_SCORE_TABLE) == EXPECTED_CELL_7A3_SCORE_SHA256),
    ('cell_7a3_score_sidecar_valid', sidecar_is_valid(CELL_7A3_SCORE_TABLE)),
    ('cell_7a3_manifest_hash_exact', sha256_file(CELL_7A3_MANIFEST) == EXPECTED_CELL_7A3_MANIFEST_SHA256),
    ('cell_7a3_manifest_sidecar_valid', sidecar_is_valid(CELL_7A3_MANIFEST)),
    ('cell_7a3_terminal_pass_exact', observed_7a3_decision == EXPECTED_CELL_7A3_DECISION),
    ('cell_7a3_score_rows_100920', score_meta['rows'] == 100_920),
    ('cell_7a3_score_columns_28', score_meta['columns'] == 28),
    ('embedding_retrieval_config_hash_exact',
     sha256_file(EMBEDDING_RETRIEVAL_CONFIG) == EXPECTED_EMBEDDING_RETRIEVAL_CONFIG_SHA256),
    ('embedding_retrieval_config_sidecar_valid',
     sidecar_is_valid(EMBEDDING_RETRIEVAL_CONFIG)),
    ('quality_config_hash_exact',
     sha256_file(QUALITY_CONFIG) == EXPECTED_QUALITY_CONFIG_SHA256),
    ('quality_config_sidecar_valid',
     sidecar_is_valid(QUALITY_CONFIG)),
    ('condition_alias_hash_exact',
     sha256_file(CONDITION_ALIASES) == EXPECTED_CONDITION_ALIASES_SHA256),
    ('condition_alias_sidecar_valid',
     sidecar_is_valid(CONDITION_ALIASES)),
    ('cell_7b4_manifest_hash_exact',
     sha256_file(CELL_7B4_MANIFEST) == EXPECTED_CELL_7B4_MANIFEST_SHA256),
    ('cell_7b4_manifest_sidecar_valid',
     sidecar_is_valid(CELL_7B4_MANIFEST)),
    ('six_blinded_aliases_frozen',
     len(alias_rows) == 6),
    ('frozen_config__cell_7b4_terminal_pass_exact', config_checks['cell_7b4_terminal_pass_exact']),
    ('frozen_config__embedding_semantic_candidate_pool_k_20', config_checks['embedding_semantic_candidate_pool_k_20']),
    ('frozen_config__embedding_final_context_k_5', config_checks['embedding_final_context_k_5']),
    ('frozen_config__embedding_same_top20_pool_for_all_conditions', config_checks['embedding_same_top20_pool_for_all_conditions']),
    ('frozen_config__embedding_candidate_pool_reused_without_mutation', config_checks['embedding_candidate_pool_reused_without_mutation']),
    ('frozen_config__embedding_hard_exclusion_false', config_checks['embedding_hard_exclusion_false']),
    ('frozen_config__rerank_input_pool_exact', config_checks['rerank_input_pool_exact']),
    ('frozen_config__rerank_input_order_semantic_ascending', config_checks['rerank_input_order_semantic_ascending']),
    ('frozen_config__rerank_final_context_k_5', config_checks['rerank_final_context_k_5']),
    ('frozen_config__rerank_hard_exclusion_false', config_checks['rerank_hard_exclusion_false']),
    ('frozen_config__rerank_quality_score_not_exposed_to_llm', config_checks['rerank_quality_score_not_exposed_to_llm']),
    ('frozen_config__rrf_formula_exact', config_checks['rrf_formula_exact']),
    ('frozen_config__rrf_semantic_weight_075', config_checks['rrf_semantic_weight_075']),
    ('frozen_config__rrf_quality_weight_025', config_checks['rrf_quality_weight_025']),
    ('frozen_config__rrf_constant_60', config_checks['rrf_constant_60']),
    ('frozen_config__rrf_descending_sort', config_checks['rrf_descending_sort']),
    ('frozen_config__condition_a_no_quality_rank', config_checks['condition_a_no_quality_rank']),
    ('frozen_config__condition_a_no_rrf', config_checks['condition_a_no_rrf']),
    ('frozen_config__condition_a_final_context_semantic_1_to_5', config_checks['condition_a_final_context_semantic_1_to_5']),
    ('frozen_config__conditions_b_to_f_rrf_applied', config_checks['conditions_b_to_f_rrf_applied']),
    ('frozen_config__conditions_b_to_f_final_context_top_five', config_checks['conditions_b_to_f_final_context_top_five']),
    ('frozen_config__random_quality_seed_20260722', config_checks['random_quality_seed_20260722']),
    ('frozen_config__random_quality_hash_sha256', config_checks['random_quality_hash_sha256']),
    ('frozen_config__random_quality_payload_exact', config_checks['random_quality_payload_exact']),
    ('frozen_config__six_condition_inventory_rows', config_checks['six_condition_inventory_rows']),
    ('frozen_config__primary_comparison_d_vs_a', config_checks['primary_comparison_d_vs_a']),
    ('authorization_targets_cell_7c2', AUTHORIZED_CELL['cell_id'] == '7C2'),
    ('expected_final_top5_rows_2400', authorization_payload['frozen_execution_design']['expected_final_top5_rows'] == 2400),
    ('hard_exclusion_prohibited', authorization_payload['frozen_execution_design']['hard_exclusion'] is False),
    ('random_seed_20260722_frozen', authorization_payload['frozen_execution_design']['random_quality_seed'] == 20260722),
    ('scores_exposed_to_llm_false', authorization_payload['frozen_execution_design']['scores_exposed_to_llm'] is False),
    ('score_rows_not_opened_in_7c1', authorization_payload['cell_7a3_score_rows_opened'] is False),
    ('answer_keys_not_inspected', authorization_payload['answer_key_outcomes_inspected'] is False),
    ('prompts_not_materialized', authorization_payload['prompts_materialized'] is False),
    ('llm_not_called', authorization_payload['llm_called'] is False),
    ('rag_metrics_not_calculated', authorization_payload['rag_metrics_calculated'] is False),
])

failed = [name for name, passed in checks.items() if not bool(passed)]
if failed:
    raise RuntimeError('Cell 7C1 authorization QC failed:\n- ' + '\n- '.join(failed))

qc_payload = {
    'cell_id': CELL_ID,
    'stage': STAGE,
    'created_utc': CREATED_UTC,
    'authorization_decision': authorization_decision,
    'passed_checks': len(checks),
    'failed_checks': 0,
    'total_checks': len(checks),
    'checks': [{'check': name, 'passed': bool(passed)} for name, passed in checks.items()],
    'scientific_operations': {
        'cell_7a3_row_level_scores_loaded': False,
        'quality_ranking_constructed': False,
        'rrf_applied': False,
        'top5_materialized': False,
        'prompts_materialized': False,
        'llm_called': False,
        'answer_keys_inspected': False,
        'rag_metrics_calculated': False,
    },
}
stable_write_json(OUTPUTS['qc'], qc_payload)
write_sidecar(OUTPUTS['qc'])

manifest_payload = {
    'cell_id': CELL_ID,
    'stage': STAGE,
    'package_version': PACKAGE_VERSION,
    'created_utc': CREATED_UTC,
    'notebook': NOTEBOOK_NAME,
    'project_root': str(ROOT),
    'output_artifacts': {
        key: {
            'path': str(path),
            'sha256': sha256_file(path),
            'sidecar_path': str(sidecar_path(path)),
            'sidecar_valid': sidecar_is_valid(path),
        }
        for key, path in OUTPUTS.items()
        if key != 'manifest'
    },
    'qc': {
        'path': str(OUTPUTS['qc']),
        'sha256': sha256_file(OUTPUTS['qc']),
        'passed_checks': len(checks),
        'failed_checks': 0,
        'total_checks': len(checks),
    },
    'authorization_decision': authorization_decision,
    'terminal_decision': (
        'PASS_STAGE7C1_COMPLETE_CELL7C0_PACKAGE_CELL7A3_SCORE_PACKAGE_AND_CELL7B4_QUALITY_'
        'CONFIGURATION_REVERIFIED_CHECKSUM_PROTECTED_CELL7C2_SCORE_LOADING_SIX_CONDITION_'
        'QUALITY_RANKING_FIXED_RRF_AND_FINAL_TOP5_MATERIALIZATION_ONLY_AUTHORIZED_NO_SCORES_'
        'OPENED_IN_AUTHORIZATION_NO_NEW_RETRIEVAL_HARD_EXCLUSION_PROMPTS_LLM_ANSWER_KEYS_'
        'ADJUDICATION_OR_RAG_METRICS'
    ),
    'next_authorized_cell': '7C2',
    'next_required_action': (
        'Execute Cell 7C2 only for checksum-frozen Cell 7A3 score loading, six-condition quality '
        'ranking, fixed 0.75/0.25 RRF with constant 60, and final top-5 materialization.'
    ),
}
stable_write_json(OUTPUTS['manifest'], manifest_payload)
write_sidecar(OUTPUTS['manifest'])

# Fresh immutable readback.
for key, path in OUTPUTS.items():
    if not path.exists():
        raise FileNotFoundError(f'Missing frozen Cell 7C1 output: {path}')
    if not sidecar_is_valid(path):
        raise AssertionError(f'Cell 7C1 output sidecar failed: {path}')

auth_readback = load_json(OUTPUTS['authorization'])
qc_readback = load_json(OUTPUTS['qc'])
manifest_readback = load_json(OUTPUTS['manifest'])

readback_checks = OrderedDict([
    ('authorization_decision_readback_exact', auth_readback['authorization_decision'] == authorization_decision),
    ('authorized_cell_readback_7c2', auth_readback['authorized_cell']['cell_id'] == '7C2'),
    ('qc_zero_failures_readback', int(qc_readback['failed_checks']) == 0),
    ('manifest_next_cell_7c2', manifest_readback['next_authorized_cell'] == '7C2'),
    ('all_four_output_sidecars_valid', all(sidecar_is_valid(path) for path in OUTPUTS.values())),
])

failed_readback = [name for name, passed in readback_checks.items() if not bool(passed)]
if failed_readback:
    raise RuntimeError('Cell 7C1 readback QC failed:\n- ' + '\n- '.join(failed_readback))

passed_checks = len(checks) + len(readback_checks)
total_checks = passed_checks

separator = '=' * 152
print('\n' + separator)
print('EXPERIMENT 2 — STAGE 7C — CELL 7C1')
print('QUALITY-RERANKING AND FINAL TOP-5 EXECUTION AUTHORIZATION FREEZE')
print(separator)
print(f'Notebook                                      : {NOTEBOOK_NAME}')
print(f'Project root                                  : {ROOT}')

print('\nUPSTREAM CELL 7C0 REVERIFICATION')
print(f'Cell 7C0 manifest SHA-256                     : {sha256_file(EXPECTED_CELL_7C0["manifest"]["path"])}')
print('Cell 7C0 terminal PASS verified               : YES')
print('Frozen Cell 7C0 artifacts                     : 10/10 exact hashes + sidecars')
print(f'Common semantic candidate rows                : {top20_meta["rows"]:,}')

print('\nFROZEN QUALITY INPUT REVERIFICATION')
print(f'Cell 7A3 score-table SHA-256                   : {sha256_file(CELL_7A3_SCORE_TABLE)}')
print(f'Cell 7A3 score-table metadata                  : {score_meta["rows"]:,} rows × {score_meta["columns"]} columns')
print('Cell 7A3 score rows opened                    : NO')
print(f'Cell 7B4 embedding-config SHA-256              : {sha256_file(EMBEDDING_RETRIEVAL_CONFIG)}')
print(f'Cell 7B4 quality-config SHA-256                : {sha256_file(QUALITY_CONFIG)}')
print(f'Cell 7B4 manifest SHA-256                      : {sha256_file(CELL_7B4_MANIFEST)}')
print('Cell 7B4 top-20 / final top-5                  : 20 / 5 structurally verified')
print('Cell 7B4 RRF                                  : 0.75 semantic + 0.25 quality; constant 60')
print(f'Cell 7B4 blinded aliases                       : {len(alias_rows)}')
print('Answer-key outcomes inspected                 : NO')

print('\nCELL 7C2 AUTHORIZATION')
print('Common candidate pool                         : frozen Cell 7C0 top-20 only')
print('Experimental conditions                      : 6')
print('Semantic-only condition                       : authorized')
print('Review/conflict-aware condition               : authorized')
print('Combined-metadata condition                   : authorized')
print('Full-GES condition                           : authorized')
print('No-star-GES condition                        : authorized')
print('Random-quality condition                     : authorized')
print('Soft rank fusion                             : 0.75 semantic + 0.25 quality RRF; constant 60')
print('Final context                                : top-5 per question-condition')
print('Expected final top-5 rows                    : 2,400')
print('Hard evidence exclusion                      : PROHIBITED')
print('Prompts / LLM                               : PROHIBITED')
print('Answer keys / metrics                       : PROHIBITED')

print('\nCELL 7C1 FROZEN OUTPUTS')
for label, path in OUTPUTS.items():
    print(f'{label:<46}: {path}')
    print(f'{"SHA-256":<46}: {sha256_file(path)}')

print(f'\nQC checks                                      : {passed_checks}/{total_checks} PASS')

print('\nSCIENTIFIC OPERATIONS IN CELL 7C1')
print('Cell 7A3 row-level scores loaded              : NO')
print('Quality ranks constructed                    : NO')
print('RRF applied                                  : NO')
print('Final top-5 contexts materialized            : NO')
print('Prompts materialized                         : NO')
print('LLM called                                   : NO')
print('Adjudication or RAG metrics                  : NO')

print('\nNEXT AUTHORIZED CELL')
print('Stage 7C — Cell 7C2                           : Frozen score loading, six-condition quality')
print('                                                 ranking, fixed RRF, and final top-5')
print('New semantic retrieval                        : PROHIBITED')
print('Prompts / LLM / answer keys / metrics         : PROHIBITED')

print(f'\nFINAL DECISION                                : {manifest_readback["terminal_decision"]}')
print(separator)



EXPERIMENT 2 — STAGE 7C — CELL 7C1
QUALITY-RERANKING AND FINAL TOP-5 EXECUTION AUTHORIZATION FREEZE
Notebook                                      : 08_GES_Aware_Genomic_RAG_Cell_7C1_Quality_Reranking_and_Top5_Execution_Authorization_V2.ipynb
Project root                                  : /content/drive/MyDrive/GES_RAG_Temporal_Study

UPSTREAM CELL 7C0 REVERIFICATION
Cell 7C0 manifest SHA-256                     : 3b645237d8d5dfa04345f649b3dd2446e94abf10fb9e855e2c98d642cdd837e8
Cell 7C0 terminal PASS verified               : YES
Frozen Cell 7C0 artifacts                     : 10/10 exact hashes + sidecars
Common semantic candidate rows                : 1,600

FROZEN QUALITY INPUT REVERIFICATION
Cell 7A3 score-table SHA-256                   : e9b162c9add5aed34a4d68d8bf625251293a18c549945468e627a498843eb802
Cell 7A3 score-table metadata                  : 100,920 rows × 28 columns
Cell 7A3 score rows opened                    : NO
Cell 7B4 embedding-config SHA-256              : ab209e